# 00 · Messages API: el loop desde cero
Modelo `claude-sonnet-5`. Datos: `workspace/contabilidad.csv` (sintético). Pregunta fija: *¿Cómo cambió el margen por proyecto de junio a julio, y por qué?*
Cuatro pasos: solo el modelo → + system prompt → + una tool → + loop.

In [1]:
import os, json, logging, warnings
from pathlib import Path

logging.getLogger("anthropic").setLevel(logging.ERROR); warnings.filterwarnings("ignore")

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))      # lee notebooks/.env si existe; no pisa variables ya exportadas

if "ANTHROPIC_API_KEY" not in os.environ:
    raise SystemExit("Falta ANTHROPIC_API_KEY en el entorno.")

MODEL = "claude-sonnet-5"
WS = (Path.cwd() if Path.cwd().name == "workspace" else Path("workspace")).resolve()  # contabilidad.csv (sintético) + .claude/skills/
(WS / "CLAUDE.md").unlink(missing_ok=True) # cada corrida empieza sin contexto de proyecto
PREGUNTA = '¿Cómo cambió el margen por proyecto de junio a julio, y por qué?'
SYSTEM = """Eres un analista financiero. Trabajas en el directorio actual, donde está ./contabilidad.csv
(separador ';', decimal con coma; columnas Period, AccountId, Debito, Credito, ProjectId, CostCenterName). No busques en otros directorios.
Reglas: ingreso = cuentas que empiezan por 4 (Credito - Debito); costo = cuentas 6 y 7 (Debito - Credito).
Margen = (ingreso - costo) / ingreso. Period 6 = junio, 7 = julio.
Verifica antes de responder. Salida: cifra por proyecto, el cálculo y la causa del cambio. Español, corto."""
SYSTEM_CORTO = 'Eres un analista financiero. Español, corto.'

import subprocess, anthropic
client = anthropic.Anthropic()
os.chdir(WS)
texto = lambda r: "".join(b.text for b in r.content if b.type == "text")

## 1 · Solo el modelo

In [2]:
r = client.messages.create(model=MODEL, max_tokens=1500, messages=[{"role": "user", "content": PREGUNTA}])
print(texto(r))

# Cambio del margen por proyecto de junio a julio

Para responder con precisión a tu pregunta, necesito que me compartas algunos datos que no tengo disponibles:

## Información que necesito:

1. **Los datos financieros** de junio y julio, como:
   - Ingresos por proyecto
   - Costos/gastos por proyecto
   - Margen de contribución o margen bruto/neto calculado

2. **Contexto del proyecto o proyectos** en cuestión:
   - Nombre o identificador del proyecto
   - Industria o tipo de negocio

## ¿Cómo puedo ayudarte mejor?

Puedes compartirme:

- 📊 Una tabla con los números de ambos meses
- 📄 Un reporte financiero (puedes pegar el texto)
- 📈 Los datos en cualquier formato (CSV, descripción textual, etc.)

**Ejemplo de formato útil:**

| Proyecto | Ingresos Jun | Costos Jun | Ingresos Jul | Costos Jul |
|----------|-------------|------------|---------------|------------|
| A | $10,000 | $6,000 | $12,000 | $7,500 |

Con esta información podré:
- Calcular el cambio exacto en el margen (%)
- Ide

## 2 · + system prompt
Sabe qué calcular, sigue sin poder leer el archivo.

In [3]:
r = client.messages.create(model=MODEL, max_tokens=1500, system=SYSTEM,
                           messages=[{"role": "user", "content": PREGUNTA}])
print(texto(r))

Voy a examinar el archivo de contabilidad.

```bash
cat ./contabilidad.csv | head -20
```

Necesito ejecutar comandos para analizar los datos. Déjame revisar la estructura y calcular los márgenes por proyecto para junio y julio.

**Nota:** Como asistente basado en texto, no tengo acceso directo a ejecutar comandos en tu sistema. Para darte la respuesta que necesitas, por favor ejecuta lo siguiente y comparte el resultado (o usa un script como el que propongo abajo):

```python
import pandas as pd

df = pd.read_csv('./contabilidad.csv', sep=';', decimal=',')

# Filtrar solo periodos 6 y 7
df = df[df['Period'].isin([6,7])]

# Clasificar cuentas
df['Tipo'] = df['AccountId'].astype(str).apply(
    lambda x: 'ingreso' if x.startswith('4') else ('costo' if x[0] in ['6','7'] else 'otro')
)

df = df[df['Tipo'] != 'otro']

# Calcular valor: ingreso = Credito - Debito; costo = Debito - Credito
df['Valor'] = df.apply(
    lambda r: (r['Credito'] - r['Debito']) if r['Tipo']=='ingreso' else (r['Deb

## 3 · + una tool (bash), sin loop
El modelo pide ejecutar un comando y se detiene: nadie lo ejecuta.

In [4]:
TOOLS = [{
    "name": "bash",
    "description": "Ejecuta un comando de shell en el directorio actual y devuelve stdout y stderr.",
    "input_schema": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]},
}]
r = client.messages.create(model=MODEL, max_tokens=1500, system=SYSTEM, tools=TOOLS,
                           messages=[{"role": "user", "content": PREGUNTA}])
print("stop_reason:", r.stop_reason)
for b in r.content:
    if b.type == "tool_use":
        print("tool_use →", b.input["command"])
print(texto(r))

stop_reason: tool_use
tool_use → cd . && head -20 contabilidad.csv && echo "---" && wc -l contabilidad.csv



## 4 · + loop
Ejecutar la tool, devolver el resultado, repetir hasta `end_turn`. Esto es un agente.

In [5]:
def bash(command: str) -> str:
    try:
        p = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=60)
    except subprocess.TimeoutExpired:
        return "error: timeout de 60 s. Usa solo ./contabilidad.csv del directorio actual."
    return (p.stdout + p.stderr)[-4000:]

messages = [{"role": "user", "content": PREGUNTA}]
for turno in range(20):
    r = client.messages.create(model=MODEL, max_tokens=2000, system=SYSTEM, tools=TOOLS, messages=messages)
    messages.append({"role": "assistant", "content": r.content})
    if r.stop_reason != "tool_use":
        break
    resultados = []
    for b in r.content:
        if b.type == "tool_use":
            print(f"→ bash: {b.input['command'][:100]}")
            resultados.append({"type": "tool_result", "tool_use_id": b.id, "content": bash(b.input["command"])})
    messages.append({"role": "user", "content": resultados})

print(f"\n[{turno + 1} turnos]")
print("".join(b.text for b in r.content if b.type == "text"))

→ bash: cd . && head -5 contabilidad.csv && echo "---" && wc -l contabilidad.csv


→ bash: cd . && python3 << 'EOF'
import csv
from collections import defaultdict

data = defaultdict(lambda: 


→ bash: cd . && python3 << 'EOF'
import csv
from collections import defaultdict

# breakdown por cuenta para



[4 turnos]
## Margen por proyecto: Junio → Julio

| Proyecto | Margen Jun | Margen Jul | Δ pp | Causa principal |
|---|---|---|---|---|
| **1030** (Almacenamiento) | 30.6% | 33.0% | **+2.4** | Ingreso +$11.3M; ahorro en arriendo bodega (-$9.3M) compensa alza en mantenimiento vehículos (+$11.3M) |
| **1045** (Almacenamiento) | 33.7% | 11.5% | **-22.2** | Costo mercancía vendida se duplicó (+$57.5M) y sueldos +$19M, pese a ahorro en arriendo (-$7.3M) |
| **2210** (Transporte) | 38.0% | 48.0% | **+10.0** | Ingreso saltó +$85M (mucho más que el aumento de mantenimiento vehículos +$25.9M) |
| **2235** (Transporte) | 30.0% | 22.0% | **-8.0** | Mantenimiento vehículos +$14.8M superó el modesto aumento de ingreso (+$3M) |
| **3310** (Transporte) | 22.7% | 19.5% | **-3.2** | Ingreso cayó -$10M y combustibles subieron +$20.6M, parcialmente compensado por menor mantenimiento (-$17.6M) |
| **3322** (Transporte) | 30.0% | 29.6% | **-0.3** | Cambios menores, prácticamente estable |
| **4410** (Tran

Modelo + tools + loop. Lo que sigue (Agent SDK, Claude Code, Managed Agents) es este loop con más palancas y sin escribirlo tú.